**Objective**

Before answering any business question through analysis, audit the core Olist datasets to evaluate data quality, identify potential issues, and understand how each table supports future business questions. This audit establishes the foundation of all future analyses.

In [2]:
# Importing customers table

import pandas as pd

customers = pd.read_csv("olist_customers_dataset.csv")

In [3]:
# How does this table look?

customers.head()

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP


In [4]:
# How many rows/columns are there?

customers.shape

(99441, 5)

**Analyst Commentary**

The dataset contains approximately 99,000 customers, which seems consistent with the size of the Olist marketplace.

In [5]:
# What are the data types?

customers.dtypes

,0
customer_id,object
customer_unique_id,object
customer_zip_code_prefix,int64
customer_city,object
customer_state,object


**Analyst Commentary**

Convert `customer_zip_code_prefix` to string.

In [6]:
# Checking for missing values

customers.isnull().sum()

,0
customer_id,0
customer_unique_id,0
customer_zip_code_prefix,0
customer_city,0
customer_state,0


**Analyst Commentary**

So, no missing values. But, if missing values existed, I would spot-check specific rows, by filtering, to see what's going on.

In [7]:
# Checking for duplicate rows

customers.duplicated().sum()

np.int64(0)

In [8]:
# Checking for duplicate primary keys

customers["customer_id"].duplicated().sum()

np.int64(0)

In [9]:
# See duplicated rows

customers[customers.duplicated()]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


**Analyst Commentary**

Primary key verified: `customer_id` is unique.

In [10]:
# Get statistical summary

customers.describe()

,customer_zip_code_prefix
count,99441.000000
mean,35137.474583
std,29797.938996
min,1003.000000
25%,11347.000000
50%,24416.000000
75%,58900.000000
max,99990.000000


In [11]:
# Get statistical summary (with objects)

customers.describe(include="object")

,customer_id,customer_unique_id,customer_city,customer_state
count,99441,99441,99441,99441
unique,99441,96096,4119,27
top,274fa6071e5e17fe303b9748641082c8,8d50f5eadf50201ccdcedfb9e2ac8455,sao paulo,SP
freq,1,17,15540,41746


**Analyst Commentary**

This table does not contain any temporal fields. Time-based analyses, such as customer acquisition trends or cohort analysis, will require joining with the orders table.

Based on the given data, there needs to be a column conversion for `customer_zip_code_prefix` because aggregation on zip codes isn't reasonable in this context.

In [12]:
# How many unique cities?

customers["customer_city"].nunique()

4119

In [13]:
# How many states?

customers["customer_state"].value_counts()

,count
customer_state,
SP,41746
RJ,12852
MG,11635
RS,5466
PR,5045
SC,3637
BA,3380
DF,2140
ES,2033


**Analyst Commentary**

All the state abbreviations look valid.

In [14]:
# Filter customer_zip_code_prefix by positive values

customers[customers["customer_zip_code_prefix"] > 0]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state
0,06b8999e2fba1a1fbc88172c00ba8bc7,861eff4711a542e4b93843c6dd7febb0,14409,franca,SP
1,18955e83d337fd6b2def6b18a428ac77,290c77bc529b7ac935b93aa66c333dc3,9790,sao bernardo do campo,SP
2,4e7b3e00288586ebd08712fdd0374a03,060e732b5b29e8181a18229c7b0b2b5e,1151,sao paulo,SP
3,b2b6027bc5c5109e529d4dc6358b12c3,259dac757896d24d7702b9acbbff3f3c,8775,mogi das cruzes,SP
4,4f2d8ab171c80ec8364f7c12e35b23ad,345ecd01c38d18a9036ed96c73b8d066,13056,campinas,SP
...,...,...,...,...,...
99436,17ddf5dd5d51696bb3d7c6291687be6f,1a29b476fee25c95fbafc67c5ac95cf8,3937,sao paulo,SP
99437,e7b71a9017aa05c9a7fd292d714858e8,d52a67c98be1cf6a5c84435bd38d095d,6764,taboao da serra,SP
99438,5e28dfe12db7fb50a4b2f691faecea5e,e9f50caf99f032f0bf3c55141f019d99,60115,fortaleza,CE
99439,56b18e2166679b8a959d72dd06da27f9,73c2643a0a458b49f58cea58833b192e,92120,canoas,RS


In [15]:
# Any negative values?

customers[customers["customer_zip_code_prefix"] < 0]

,customer_id,customer_unique_id,customer_zip_code_prefix,customer_city,customer_state


In [16]:
# Any absurd values?

customers["customer_zip_code_prefix"].max()

99990

In [17]:
# Average length of Customer IDs

customers["customer_id"].str.len().mean()

np.float64(32.0)

**Audit Summary**

**Table:** Customers

**Purpose:** Stores customer location and unique customer identifiers

**Primary Key:** `customer_id`

**Row Count:** 99,441

**Missing Values:** None

**Duplicate Rows:** None

**Duplicate Primary Keys:** None

**Potential Issues:**

1. Zip code stored as an integer
2. Need to validate state abbreviations
3. Need to validate city spellings

**Cleaning Recommendation:** Convert zip code to string/object.

No additional cleaning required.

**Business Questions This Table Can Help Answer**:

1. Which geographic regions have the largest customer base?

2. Which geographic regions are not receiving deliveries on time?

**Analyst Commentary**

Next, I audit the Orders table because it contains the transactional events that will support the operational and customer experience analyses later in this project.

In [18]:
# Importing orders table

import pandas as pd

orders = pd.read_csv("olist_orders_dataset.csv")

In [19]:
# How does this table look?

orders

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
0,e481f51cbdc54678b7cc49136f2d6af7,9ef432eb6251297304e76186b10a928d,delivered,2017-10-02 10:56:33,2017-10-02 11:07:15,2017-10-04 19:55:00,2017-10-10 21:25:13,2017-10-18 00:00:00
1,53cdb2fc8bc7dce0b6741e2150273451,b0830fb4747a6c6d20dea0b8c802d7ef,delivered,2018-07-24 20:41:37,2018-07-26 03:24:27,2018-07-26 14:31:00,2018-08-07 15:27:45,2018-08-13 00:00:00
2,47770eb9100c2d0c44946d9cf07ec65d,41ce2a54c0b03bf3443c3d931a367089,delivered,2018-08-08 08:38:49,2018-08-08 08:55:23,2018-08-08 13:50:00,2018-08-17 18:06:29,2018-09-04 00:00:00
3,949d5b44dbf5de918fe9c16f97b45f8a,f88197465ea7920adcdbec7375364d82,delivered,2017-11-18 19:28:06,2017-11-18 19:45:59,2017-11-22 13:39:59,2017-12-02 00:28:42,2017-12-15 00:00:00
4,ad21c59c0840e6cb83a9ceb5573f8159,8ab97904e6daea8866dbdbc4fb7aad2c,delivered,2018-02-13 21:18:39,2018-02-13 22:20:29,2018-02-14 19:46:34,2018-02-16 18:17:02,2018-02-26 00:00:00
...,...,...,...,...,...,...,...,...
99436,9c5dedf39a927c1b2549525ed64a053c,39bd1228ee8140590ac3aca26f2dfe00,delivered,2017-03-09 09:54:05,2017-03-09 09:54:05,2017-03-10 11:18:03,2017-03-17 15:08:01,2017-03-28 00:00:00
99437,63943bddc261676b46f01ca7ac2f7bd8,1fca14ff2861355f6e5f14306ff977a7,delivered,2018-02-06 12:58:58,2018-02-06 13:10:37,2018-02-07 23:22:42,2018-02-28 17:37:56,2018-03-02 00:00:00
99438,83c1379a015df1e13d02aae0204711ab,1aa71eb042121263aafbe80c1b562c9c,delivered,2017-08-27 14:46:43,2017-08-27 15:04:16,2017-08-28 20:52:26,2017-09-21 11:24:17,2017-09-27 00:00:00
99439,11c177c8e97725db2631073c19f07b62,b331b74b18dc79bcdf6532d51e1637c1,delivered,2018-01-08 21:28:27,2018-01-08 21:36:21,2018-01-12 15:35:03,2018-01-25 23:32:54,2018-02-15 00:00:00


In [20]:
# How many rows/columns are there?

orders.shape

(99441, 8)

**Analyst Commentary**

The dataset contains approximately 99,000 orders, which seems consistent with the size of the Olist marketplace.

In [21]:
# What are the data types?

orders.dtypes

,0
order_id,object
customer_id,object
order_status,object
order_purchase_timestamp,object
order_approved_at,object
order_delivered_carrier_date,object
order_delivered_customer_date,object
order_estimated_delivery_date,object


**Analyst Commentary**

Primary key verified: `order_id` is unique.

All of the columns containing dates and times are stored as strings. These fields should be converted to `datetime` so that delivery performance and other time-based analyses can be performed accurately.

In [23]:
# Checking for missing values

orders.isnull().sum()

,0
order_id,0
customer_id,0
order_status,0
order_purchase_timestamp,0
order_approved_at,160
order_delivered_carrier_date,1783
order_delivered_customer_date,2965
order_estimated_delivery_date,0


**Analyst Commentary**

There are missing values in some columns, which can impact time-series analysis. This will be investigated during the data cleaning phase.

In [24]:
# Checking for duplicate rows

orders.duplicated().sum()

np.int64(0)

In [25]:
# Checking for duplicate primary keys

orders["order_id"].duplicated().sum()

np.int64(0)

**Analyst Commentary**

If we were checking for duplicates on customer_id in the orders table, that's ok. A customer may place many orders, so the same customer_id can legitimately appear multiple times.

In [26]:
# See duplicated rows

orders[orders.duplicated()]


,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date


In [27]:
# Get statistical summary

orders.describe()

,order_id,customer_id,order_status,order_purchase_timestamp,order_approved_at,order_delivered_carrier_date,order_delivered_customer_date,order_estimated_delivery_date
count,99441,99441,99441,99441,99281,97658,96476,99441
unique,99441,99441,8,98875,90733,81018,95664,459
top,66dea50a8b16d9b4dee7af250b4be1a5,edb027a75a1449115f6b43211ae02a24,delivered,2018-08-02 12:06:07,2018-02-27 04:31:10,2018-05-09 15:48:00,2018-05-14 20:02:44,2017-12-20 00:00:00
freq,1,1,96478,3,9,47,3,522


In [28]:
# How many unique statuses?

orders["order_status"].nunique()

8

In [29]:
# What are those statuses and their totals?

orders["order_status"].value_counts()

,count
order_status,
delivered,96478
shipped,1107
canceled,625
unavailable,609
invoiced,314
processing,301
created,5
approved,2


**Analyst Commentary**

Are these status mutually exclusive?

The overwhelming majority of orders are delivered, suggesting the platform successfully fulfills most purchases. However, the remaining statuses may warrant investigation to understand operational bottlenecks or customer experience issues.

**Audit Summary**

**Table**: Orders

**Purpose**: Stores customer order transactions and fulfillment timestamps

**Primary Key**: `order_id`

**Row Count**: 99,441

**Missing Values**: Yes

**Duplicate Rows**: No

**Duplicate Primary Keys**: None

**Potential Issues**:

1. `order_purchase_timestamp`, `order_approved_at`, `order_delivered_carrier_date`, `order_delivered_customer_date`, and `order_estimated_delivery_date` are stored as a string. Convert to `datetime`.

2. `order_approved_at`, `order_delivered_carrier_date`, and `order_delivered_customer_date` contain missing values. This will impact time-series analysis.

**Cleaning Recommendation**

1. Convert indicated columns in potential issues to datetime

2. Investigate the missing values of the indicated columns in potential issues during the cleaning phase

No additional cleaning required.

**Business Questions This Table Can Help Answer**:

1. Which order statuses may require operational investigation?

2. What's the average number of days for delayed orders?